# Turbulence Benchmark Database Builder

The Turbulence benchmark is used to evaluate the robustness of LLMs in code generation tasks. This notebook uses the source code from the benchmark and adapts it for the use case of testing with MuCoCo.

In [1]:
import os
import random
from typing import Iterable
from tqdm import tqdm
import sys

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from utility.helper_functions import TurbulenceBenchmarkHelper
from database import MongoDBHelper

# Connecting to MongoDB

In [4]:
db = MongoDBHelper(max_retries= 5)
if db.check_database_connectivity():
    print("MongoDB connected")

base_qns_db = db.client["Baseline_Questions_DB"]
baseline_db = base_qns_db["Turbulence_Benchmark"]

MongoDB connected


In [5]:
curr_dir = os.getcwd()
source_code_dir = os.path.join(curr_dir, "Source_Code")
qn_folders = [f for f in os.listdir(source_code_dir) if os.path.isdir(os.path.join(source_code_dir, f))]

In [ ]:
seed = 1234
random.seed(seed)

failed_tasks = []
for qn_folder_name in tqdm(qn_folders):
    # setting the correct qn_num
    q_no = qn_folder_name.split("Q")[-1]  
    if q_no != "":
        
        func_name = None            # stores the task function name
        
        # obtaining the folder directory to the target qn. In this benchmark, each question is kept in an individual folder.
        qn_folder_dir = os.path.join(source_code_dir, qn_folder_name)
        
        # initializing a TurbulenceBenchmarkHelper object with the question number and seed
        helper = TurbulenceBenchmarkHelper(q_no= q_no, seed=seed)

        # 1. Generating the params to substitute into the solutions etc. 
        gen_params_res = helper.run_gen_params(
            qn_folder_dir = qn_folder_dir,
        )

        # Ensuring that each parameter generated is iterable, else it is converted to a Tuple.
        # This step is necessary as some test functions require an iterable input
        gen_params_res = [(param, ) if not isinstance(param, Iterable) else param for param in gen_params_res]

        # 2. Using the generated params to generate the function inputs.
        input_generator_res = helper.run_input_generator(
            qn_folder_dir = qn_folder_dir,
            gen_params = gen_params_res
        )

        # 3. Obtaining the solution, natural language and test templates
        sol_template = TurbulenceBenchmarkHelper.return_template_contents(
            dir = os.path.join(qn_folder_dir, "solution.py.template")
        )

        prompt_template = TurbulenceBenchmarkHelper.return_template_contents(
            dir = os.path.join(qn_folder_dir, "question.txt.template")
        )

        tests_template = TurbulenceBenchmarkHelper.return_template_contents(
            dir = os.path.join(qn_folder_dir, "tests.py.template")
        )

        # Removing the unnecessary import statements in test template
        tests_template = helper.process_test_cases(test_template=tests_template)

        # iterating through each sample generated for each task
        # this step involves iterating through each 
        for idx in range(len(input_generator_res)):
            params = gen_params_res[idx]
            func_input = input_generator_res[idx]

            solution = sol_template
            prompt = prompt_template
            tests = tests_template

            for param_idx, param in enumerate(params):
                solution = solution.replace(f"${param_idx}", str(param))
                tests = tests.replace(f"${param_idx}", str(param))
            
            if func_name is None:
                try:
                    func_name = helper.obtain_func_name(
                        sol_template= solution,
                        qn_txt_template= prompt_template
                        )
                except Exception as e:
                    print(e)
                    failed_tasks.append((q_no, e))
                    continue
            
            
            tests = helper.replace_func_name(tests_template = tests, func_name = func_name)

            try:
                helper.run_tests(tests = tests, solution = solution, func_name= func_name)
            except Exception as e:
                print(f'Q{q_no} failed the tests and will not be added to the Turbulence database')
                failed_tasks.append(q_no)
                break
        
        # If statement checking if the question has failed the checks. If so, continue to the next task and do not store this task in the database. 
        if q_no in failed_tasks:
            continue

        # Parameter and test inputs dictionary
        param_dict = {}
        for idx in range(len(gen_params_res)):
            params_entry = gen_params_res[idx]
            input_entry = input_generator_res[idx]

            param_dict[str(idx)] = {
                "params": helper.process_for_mongo_db_storage(params_entry),
                "func_input" : helper.process_for_mongo_db_storage(input_entry)
            }
        

        # Creating the database entry
        database_entry = {
            "_id": f"TurbulenceQ{q_no}",
            "question_template": tests_template,
            "prompt_template": prompt_template,
            "solution_template": sol_template,
            "func_name": func_name,
            "params": param_dict
        }
        
        # Storing entry into the baseline database
        exisiting_entry = baseline_db.find_one({"_id": f"TurbulenceQ{q_no}"})

        try:
            if exisiting_entry is not None:
                baseline_db.find_one_and_replace({"_id": f"TurbulenceQ{q_no}"}, database_entry)
            else:
                baseline_db.insert_one(database_entry)
        except Exception as e:
            print(e)
            continue


        # TODO: continue with the retrieval and testing again
    

 20%|██        | 12/60 [02:28<08:23, 10.49s/it]

adding Q8 to failed_tasks


 30%|███       | 18/60 [03:26<05:32,  7.92s/it]

adding Q13 to failed_tasks


 33%|███▎      | 20/60 [03:36<04:05,  6.14s/it]

...

adding Q21 to failed_tasks


 92%|█████████▏| 55/60 [10:02<00:35,  7.19s/it]

adding Q29 to failed_tasks


100%|██████████| 60/60 [10:45<00:00, 10.75s/it]

adding Q42 to failed_tasks


### Printing all tasks that failed

In [12]:
print(f"There are a total of {len(failed_tasks)}/60 total tasks were not added to the Turbulence database")
print("The following tasks failed the tests:")
for t in failed_tasks:
    print(f"    Q{t}")

There are a total of 8/60 total tasks were not added to the Turbulence database
The following tasks failed the tests:
    Q8
    Q13
    Q22
    Q41
    Q48
    Q21
    Q29
    Q42


In [ ]:
num_tasks = baseline_db.count_documents({})
